# Xatu Beacon Chain Analysis

This notebook demonstrates how to use the xatu-analysis library to analyze beacon chain data from Xatu.


In [ ]:
# Import the xatu-analysis library
import sys
sys.path.append('../src')

from xatu_analysis import (
    ClickHouseConnector,
    ParquetConnector,
    BeaconEvent,
    create_time_series_plot,
    create_histogram
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Configuration

Set up your connection to ClickHouse or use Parquet files. 

For ClickHouse, you can either:
1. Set environment variables (CLICKHOUSE_HOST, CLICKHOUSE_PORT, etc.)
2. Create a `config.json` file in the project root
3. Pass connection parameters directly to the connector


In [ ]:
# Option 1: Using ClickHouse (uncomment and configure as needed)
# ch_connector = ClickHouseConnector(
#     host="your-clickhouse-host",
#     port=8123,
#     username="default",
#     password="your-password",
#     database="xatu"
# )

# Option 2: Using Parquet files
parquet_connector = ParquetConnector(base_path="../data")

print("Connectors initialized successfully!")

## Loading Beacon Chain Data

Load beacon chain events from either ClickHouse or Parquet files.


In [ ]:
# Load data from ClickHouse (if available)
# beacon_data = ch_connector.get_beacon_events(
#     start_time="2024-01-01 00:00:00",
#     end_time="2024-01-02 00:00:00",
#     limit=10000
# )

# Load data from Parquet files (example with mock data)
try:
    beacon_data = parquet_connector.get_beacon_events_from_parquet(
        date_pattern="beacon_events_*.parquet",
        start_time="2024-01-01",
        end_time="2024-01-02"
    )
    print(f"Loaded {len(beacon_data)} beacon events")
    print(beacon_data.head())
except FileNotFoundError:
    print("No parquet files found. Creating sample data for demonstration...")
    
    # Create sample data for demonstration
    sample_data = {
        'event_date_time': pd.date_range('2024-01-01', periods=1000, freq='12s'),
        'slot': range(1000),
        'epoch': [i // 32 for i in range(1000)],
        'proposer_index': np.random.randint(0, 500000, 1000),
        'meta_client_name': np.random.choice(['lighthouse', 'prysm', 'teku', 'nimbus'], 1000),
        'meta_network_name': 'mainnet'
    }
    beacon_data = pd.DataFrame(sample_data)
    print(f"Created {len(beacon_data)} sample beacon events")
    print(beacon_data.head())

## Basic Analysis

Perform basic analysis on the beacon chain data.


In [ ]:
# Basic statistics
print("Dataset Info:")
print(f"Total events: {len(beacon_data)}")
print(f"Date range: {beacon_data['event_date_time'].min()} to {beacon_data['event_date_time'].max()}")
print(f"Unique slots: {beacon_data['slot'].nunique()}")
print(f"Unique epochs: {beacon_data['epoch'].nunique()}")
print(f"Client distribution:")
print(beacon_data['meta_client_name'].value_counts())

## Visualizations

Create visualizations using the built-in plotting functions.


In [ ]:
# Create a time series plot of events per hour
hourly_events = beacon_data.set_index('event_date_time').resample('H').size().reset_index()
hourly_events.columns = ['hour', 'event_count']

create_time_series_plot(
    hourly_events,
    x_col='hour',
    y_col='event_count',
    title='Beacon Events per Hour',
    interactive=False
)

In [ ]:
# Distribution of proposer indices
create_histogram(
    beacon_data,
    col='proposer_index',
    bins=50,
    title='Distribution of Proposer Indices',
    interactive=False
)

In [ ]:
# Client distribution pie chart
client_counts = beacon_data['meta_client_name'].value_counts()

plt.figure(figsize=(10, 8))
plt.pie(client_counts.values, labels=client_counts.index, autopct='%1.1f%%')
plt.title('Beacon Client Distribution')
plt.show()

## Advanced Analysis

Perform more advanced analysis using the data models.


In [ ]:
# Convert to BeaconEvent objects for advanced analysis
beacon_events = []
for _, row in beacon_data.head(100).iterrows():  # Just first 100 for performance
    event = BeaconEvent(
        event_date_time=row['event_date_time'],
        meta_client_name=row['meta_client_name'],
        meta_client_id=f"client_{row.name}",
        meta_client_version="1.0.0",
        meta_client_implementation=row['meta_client_name'],
        meta_network_name=row.get('meta_network_name', 'mainnet'),
        slot=row['slot'],
        epoch=row['epoch'],
        proposer_index=row['proposer_index']
    )
    beacon_events.append(event)

print(f"Created {len(beacon_events)} BeaconEvent objects")

# Use computed properties
for event in beacon_events[:5]:
    print(f"Slot {event.slot}: Epoch {event.epoch_from_slot}, Slot Time: {event.slot_time}")

## Export Results

Save processed data for future use.


In [ ]:
# Save processed data
output_file = "processed_beacon_events.parquet"
parquet_connector.write_parquet(beacon_data, output_file)
print(f"Saved processed data to {output_file}")

# Also save as CSV for compatibility
beacon_data.to_csv("processed_beacon_events.csv", index=False)
print("Saved processed data to processed_beacon_events.csv")